# SentinelAI — Network Intrusion Detection (NSL-KDD)

**Purpose:** build and evaluate a binary classifier that predicts whether a network connection is `normal` or an `attack`.

**Models retained from the original notebook**
- Random Forest with hyperparameter tuning
- Logistic Regression
- Random Forest with class weights
- Random Forest with SMOTE (optional, requires `imbalanced-learn`)
- Probability-threshold comparison

**Important evaluation rule:** model selection and threshold selection use training/validation data. The test set is evaluated at the end and should not be used to choose the threshold. Metrics from NSL-KDD are benchmark results, not proof of real-world production performance.

## 1. Environment

Install the packages in your project environment if needed:

```bash
python -m pip install pandas numpy matplotlib seaborn scikit-learn requests joblib
python -m pip install imbalanced-learn
```

The SMOTE experiment is optional. If `imbalanced-learn` is not installed, the rest of the notebook still runs.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.base import clone
import joblib
import sklearn

RANDOM_STATE = 42
print("scikit-learn:", sklearn.__version__)

## 2. Load NSL-KDD

The dataset is downloaded from the public `defcom17/NSL_KDD` GitHub repository. If the download fails, download `KDDTrain+.txt` and `KDDTest+.txt` manually and update the paths below.

In [ ]:
TRAIN_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
TEST_URL  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"

COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty"
]

train_raw = pd.read_csv(TRAIN_URL, header=None, names=COLUMNS)
test_raw = pd.read_csv(TEST_URL, header=None, names=COLUMNS)

print("Training rows/columns:", train_raw.shape)
print("Test rows/columns:", test_raw.shape)
display(train_raw.head())

## 3. Explore and prepare the labels

NSL-KDD contains multiple attack labels. For this notebook, they are grouped into two classes:
- `0 = normal`
- `1 = attack`

The `difficulty` field is removed because it is dataset metadata rather than an input feature.

In [ ]:
for name, df in [("Train", train_raw), ("Test", test_raw)]:
    print(f"\n{name} label counts:")
    display(df["label"].value_counts().head(15).to_frame("count"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_raw["label"].map(lambda x: "normal" if x == "normal" else "attack").value_counts().plot(
    kind="bar", ax=axes[0], title="Training set: normal vs attack"
)
test_raw["label"].map(lambda x: "normal" if x == "normal" else "attack").value_counts().plot(
    kind="bar", ax=axes[1], title="Test set: normal vs attack"
)
for ax in axes:
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of records")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
def prepare_xy(df):
    df = df.drop(columns=["difficulty"], errors="ignore").copy()
    y = (df["label"] != "normal").astype("int8")
    X = df.drop(columns=["label"])
    return X, y

X_train_full, y_train_full = prepare_xy(train_raw)
X_test, y_test = prepare_xy(test_raw)

print("Training class proportions:")
display(y_train_full.value_counts(normalize=True).rename(index={0:"normal", 1:"attack"}).to_frame("proportion"))
print("Test class proportions:")
display(y_test.value_counts(normalize=True).rename(index={0:"normal", 1:"attack"}).to_frame("proportion"))

numeric_features = X_train_full.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_full.select_dtypes(exclude=["number"]).columns.tolist()
print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

## 4. Train/validation split and leakage-safe preprocessing

The official test set is kept aside. A validation subset is taken only from the training set to select the probability threshold. Imputation, scaling, one-hot encoding, and feature selection are fitted inside each pipeline, so cross-validation does not learn preprocessing from its validation folds.

`StandardScaler` is included for the shared preprocessing pipeline. It is useful for Logistic Regression; Random Forest does not depend on feature scaling.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Train split:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

## 5. Random Forest + feature selection + hyperparameter tuning

The feature selector is part of the pipeline, so it is fitted only on the relevant training fold during cross-validation. The search optimizes attack recall, because missed attacks are costly in intrusion detection. Precision and false-positive rate will also be reported; recall alone is not sufficient to choose a model.

In [ ]:
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        threshold="median"
    )),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=3,
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print("Best mean CV attack recall:", round(grid.best_score_, 4))

## 6. Compare Random Forest and Logistic Regression on validation data

This comparison is based on the same training/validation split. The selected Random Forest pipeline includes feature selection; Logistic Regression uses the same preprocessing without that selector.

In [ ]:
lr_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, class_weight=None, random_state=RANDOM_STATE))
])
lr_pipeline.fit(X_train, y_train)

def metrics_at_threshold(y_true, probabilities, threshold=0.5):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "attack_precision": precision_score(y_true, pred, zero_division=0),
        "attack_recall": recall_score(y_true, pred, zero_division=0),
        "attack_f1": f1_score(y_true, pred, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0,
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities)
    }

val_prob_rf = best_rf.predict_proba(X_val)[:, 1]
val_prob_lr = lr_pipeline.predict_proba(X_val)[:, 1]

validation_comparison = pd.DataFrame([
    {"model": "Random Forest tuned", **metrics_at_threshold(y_val, val_prob_rf, 0.5)},
    {"model": "Logistic Regression", **metrics_at_threshold(y_val, val_prob_lr, 0.5)}
])
display(validation_comparison.round(4))

## 7. Threshold selection using validation data only

A lower threshold usually increases attack recall but can also increase false alarms. This cell compares thresholds on the validation split, not the test set. Choose the threshold based on the use case and the acceptable false-positive rate. The example below selects the threshold with the highest recall among thresholds whose validation false-positive rate is at most 5%; if none meets that condition, it selects the threshold with the lowest validation false-positive rate.

In [ ]:
thresholds = np.arange(0.10, 0.81, 0.05)
threshold_rows = []
for t in thresholds:
    threshold_rows.append(metrics_at_threshold(y_val, val_prob_rf, float(round(t, 2))))
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table[[
    "threshold", "attack_precision", "attack_recall", "attack_f1",
    "false_positive_rate", "false_negatives", "false_positives"
]].round(4))

acceptable = threshold_table[threshold_table["false_positive_rate"] <= 0.05]
if not acceptable.empty:
    chosen_threshold = float(acceptable.sort_values(
        ["attack_recall", "attack_precision"], ascending=False
    ).iloc[0]["threshold"])
else:
    chosen_threshold = float(threshold_table.sort_values(
        ["false_positive_rate", "attack_recall"], ascending=[True, False]
    ).iloc[0]["threshold"])

print("Threshold selected from validation data:", chosen_threshold)

plt.figure(figsize=(8, 4))
plt.plot(threshold_table["threshold"], threshold_table["attack_recall"], marker="o", label="Attack recall")
plt.plot(threshold_table["threshold"], threshold_table["attack_precision"], marker="o", label="Attack precision")
plt.plot(threshold_table["threshold"], threshold_table["false_positive_rate"], marker="o", label="False-positive rate")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Validation threshold trade-off")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 8. Class-weighted Random Forest

This experiment keeps the original class-weight idea. The model is trained on the training split and evaluated on validation data, using the same preprocessing and feature-selection strategy.

In [ ]:
weighted_rf = clone(rf_pipeline).set_params(
    model__class_weight={0: 1, 1: 2}
)
weighted_rf.fit(X_train, y_train)
weighted_prob = weighted_rf.predict_proba(X_val)[:, 1]

weighted_comparison = pd.DataFrame([
    {"model": "Random Forest tuned", **metrics_at_threshold(y_val, val_prob_rf, chosen_threshold)},
    {"model": "Random Forest class_weight {0:1, 1:2}", **metrics_at_threshold(y_val, weighted_prob, chosen_threshold)}
])
display(weighted_comparison.round(4))

## 9. Optional SMOTE experiment

SMOTE must be applied only to training data, never to validation or test data. Because one-hot encoded categorical values can be problematic for ordinary SMOTE, this experiment uses `SMOTENC` on the raw training data, which explicitly identifies categorical columns. It is optional and can be slow. If the package is unavailable, skip this section.

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTENC

    categorical_indices = [X_train.columns.get_loc(c) for c in categorical_features]
    smote_nc = SMOTENC(
        categorical_features=categorical_indices,
        random_state=RANDOM_STATE,
        sampling_strategy="auto"
    )
    # SMOTENC operates on raw columns. The following pipeline oversamples
    # only the training folds, then applies preprocessing and classification.
    smote_pipeline = ImbPipeline([
        ("smote", smote_nc),
        ("preprocess", clone(preprocessor)),
        ("select", SelectFromModel(
            RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
            threshold="median"
        )),
        ("model", RandomForestClassifier(
            n_estimators=200, max_depth=None, min_samples_split=2,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
    smote_pipeline.fit(X_train, y_train)
    smote_prob = smote_pipeline.predict_proba(X_val)[:, 1]
    smote_metrics = metrics_at_threshold(y_val, smote_prob, chosen_threshold)
    print("SMOTE validation metrics:")
    display(pd.DataFrame([{"model": "Random Forest + SMOTENC", **smote_metrics}]).round(4))
except ImportError:
    smote_pipeline = None
    print("imbalanced-learn is not installed. Optional SMOTE experiment skipped.")
except Exception as e:
    smote_pipeline = None
    print("Optional SMOTE experiment could not run:", repr(e))

## 10. Final test evaluation

Only after selecting the model and threshold on training/validation data do we evaluate on the untouched official test set. This is the primary final evaluation. Do not change the threshold after viewing these test results.

In [ ]:
# Refit the selected Random Forest configuration on the full official training set.
final_model = clone(best_rf)
final_model.fit(X_train_full, y_train_full)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred_05 = (test_prob >= 0.5).astype(int)
test_pred_selected = (test_prob >= chosen_threshold).astype(int)

print("FINAL TEST RESULTS — default threshold 0.50")
print(classification_report(y_test, test_pred_05, target_names=["normal", "attack"], zero_division=0))
print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, test_pred_05))
print(pd.DataFrame([metrics_at_threshold(y_test, test_prob, 0.5)]).round(4).to_string(index=False))

print("\nFINAL TEST RESULTS — validation-selected threshold", chosen_threshold)
print(classification_report(y_test, test_pred_selected, target_names=["normal", "attack"], zero_division=0))
print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, test_pred_selected))
final_test_metrics = metrics_at_threshold(y_test, test_prob, chosen_threshold)
display(pd.DataFrame([final_test_metrics]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred_selected, display_labels=["normal", "attack"],
    cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title(f"Confusion matrix (threshold={chosen_threshold:.2f})")
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[1], name="Random Forest")
axes[1].set_title("ROC curve — official test set")
plt.tight_layout()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, test_prob, name="Random Forest")
plt.title("Precision–Recall curve — official test set")
plt.show()

## 11. Feature importance

Feature names are retrieved from the fitted preprocessing pipeline and then filtered by the fitted feature selector. Importance indicates how the trained Random Forest uses features; it does not prove causal relationships.

In [ ]:
pre = final_model.named_steps["preprocess"]
selector = final_model.named_steps["select"]
rf = final_model.named_steps["model"]

try:
    transformed_names = pre.get_feature_names_out()
    selected_names = transformed_names[selector.get_support()]
    importances = rf.feature_importances_
    importance_df = pd.DataFrame({
        "feature": selected_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    display(importance_df.head(20))
    top = importance_df.head(15).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(top["feature"], top["importance"])
    plt.title("Top 15 selected feature importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Feature importance display could not be generated:", repr(e))

## 12. Save the final model and evaluation summary

The saved pipeline includes preprocessing, feature selection, and the classifier. Save the chosen threshold separately because the threshold is part of the decision policy.

In [ ]:
MODEL_PATH = "sentinelai_nsl_kdd_random_forest.joblib"
THRESHOLD_PATH = "sentinelai_threshold.txt"
METRICS_PATH = "sentinelai_test_metrics.csv"

joblib.dump(final_model, MODEL_PATH)
with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    f.write(str(chosen_threshold))

pd.DataFrame([{
    "dataset": "NSL-KDD",
    "model": "Random Forest with SelectFromModel feature selection",
    "threshold_selected_on_validation": chosen_threshold,
    **final_test_metrics
}]).to_csv(METRICS_PATH, index=False)

print("Saved:", MODEL_PATH, THRESHOLD_PATH, METRICS_PATH)

## 13. Conclusions and limitations

Use the actual values printed in the final test evaluation when writing the report. Do not copy earlier metrics from the original notebook because preprocessing, data splitting, and threshold selection have changed.

Discuss:
1. Attack recall: what proportion of attacks were detected?
2. Attack precision: how many predicted attacks were real attacks?
3. False-positive rate: how often was normal traffic incorrectly flagged?
4. The threshold trade-off and why the threshold was selected on validation data.
5. Limitations: NSL-KDD is an older benchmark; real-time use requires compatible flow-feature extraction, latency testing, and evaluation on representative current traffic.

**Do not claim real-time detection or production readiness yet.** This notebook evaluates offline records. A next milestone is to build a prediction API and a dashboard, then integrate a validated traffic feature extractor.